# IND320 Project Work – Part 1

### Project links

- **By:** Agop Haroun
- **GitHub repository:** [G0KU-DOTCOM/Streamlit_dashboard](https://github.com/G0KU-DOTCOM/Streamlit_dashboard)
- **Streamlit application:** Not deployed yet. The public link will be added later.

### AI usage

I used ChatGPT as a learning and review tool while working on this part of
the project. It helped me interpret the assignment, discuss understandable English column names, and improve the
structure and comments in the notebook. It also suggested ways to investigate
the unusual year-1 publication date and compare measurements with different scales.
I reviewed the suggestions by running the code and comparing them with the CSV
outputs. AI did not create or modify the source data, and uncertain
interpretations are identified as interpretations rather than documented facts.


In this notebook, I will prepare and explore the Norwegian reservoir
dataset before using the same data in my Streamlit app.

## 1. Importing the reservoir data

The CSV file is stored in the `data` folder, while this notebook is stored
in `notebooks`. I use a relative path instead of the full path on my
computer so the notebook can still find the data when the repository is
downloaded or cloned by someone else.

In [ ]:
# I keep all imports in one cell so the notebook dependencies are easy to find.
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display

In [ ]:
# The two dots take me one folder back from notebooks/.
# From there, I enter the data folder and read the CSV file.
reservoirs = pd.read_csv("../data/reservoirs.csv")

# I print the dimensions first to confirm that the complete file was loaded.
print(f"The dataset contains {reservoirs.shape[0]:,} rows and {reservoirs.shape[1]} columns.")

# I create a small overview of the original headers, their data types,
# and one example value. This helps me understand what each column represents
# before I decide on clearer English names.
header_review = pd.DataFrame({
    "original_header": reservoirs.columns,
    "data_type": reservoirs.dtypes.astype(str).values,
    "example_value": [
        reservoirs[column].iloc[0]
        for column in reservoirs.columns
    ]
})

header_review

## 2. Understanding and renaming the columns

Before changing the headers, I compare their original names, data types, and
example values. I then rename only the headers. The original observations and
category values remain unchanged.

In [ ]:
# I rename the original Norwegian headers to make the dataset easier to
# understand when I create tables, plots, and the Streamlit application.
english_names = {
    "dato_Id": "observation_date",
    "omrType": "area_type",
    "omrnr": "area_number",
    "iso_aar": "iso_year",
    "iso_uke": "iso_week",
    "fyllingsgrad": "filling_fraction",
    "kapasitet_TWh": "capacity_twh",
    "fylling_TWh": "stored_energy_twh",
    "neste_Publiseringsdato": "next_publication_date",
    "fyllingsgrad_forrige_uke": "previous_week_filling_fraction",
    "endring_fyllingsgrad": "weekly_filling_change"
}

reservoirs = reservoirs.rename(columns=english_names)

# I print the updated headers to check that every column was renamed correctly.
print(reservoirs.columns.tolist())

### Explanation of the renamed columns

I renamed the original Norwegian headers to make their meanings clearer and to
use consistent English names throughout the notebook and Streamlit application.

- **observation_date:** The date on which the reservoir level was recorded.
- **area_type:** A code used to separate the geographical groupings in the
  dataset. I keep the original values `EL`, `VASS`, and `NO` unchanged.
- **area_number:** Identifies the numbered area within the selected area type.
- **iso_year:** The year according to the ISO calendar.
- **iso_week:** The week number according to the ISO calendar.
- **filling_fraction:** The proportion of the total reservoir capacity currently
  filled. For example, `0.80` means that the reservoirs are 80% full.
- **capacity_twh:** The maximum energy-storage capacity of the reservoirs,
  measured in terawatt-hours (TWh).
- **stored_energy_twh:** The amount of energy currently stored in the reservoirs,
  measured in TWh.
- **next_publication_date:** The next publication date stored in the dataset.
- **previous_week_filling_fraction:** The filling fraction recorded during the
  previous week.
- **weekly_filling_change:** The change in filling fraction compared with the
  previous week. For example, `0.01` corresponds to an increase of one
  percentage point.

The category values have not been renamed because their exact definitions are
not included with the CSV file. Keeping them unchanged prevents me from adding
a meaning that cannot be confirmed from the dataset itself.

## 3. Displaying and checking the data

Printing all 14,877 rows would make the notebook difficult to read. I first
print the date range and then use compact summaries to check the structure,
missing values, duplicates, and unusual values. A larger chronological sample
is shown afterwards when I investigate the publication-date column.

In [ ]:
# The CSV is not arranged chronologically, so I check the complete date range
# before sorting and examining the observations in more detail.
first_date = reservoirs["observation_date"].min()
last_date = reservoirs["observation_date"].max()

print(f"First observation date: {first_date}")
print(f"Latest observation date: {last_date}")

### Data quality and area structure

The following summary checks the data types, missing values, unique values,
duplicates, and the relationship between area types and area numbers.

In [ ]:
# Pandas only counts formal missing values such as NaN or None. I therefore
# count the unusual year-1 date separately without deciding what it means.
year_one_date = "0001-01-01T00:00:00"

# This compact table lets me review the structure without printing the full CSV.
column_summary = pd.DataFrame({
    "data_type": reservoirs.dtypes.astype(str),
    "pandas_missing_values": reservoirs.isna().sum(),
    "unique_values": reservoirs.nunique()
})

column_summary["year_one_date_values"] = 0
column_summary.loc["next_publication_date", "year_one_date_values"] = (
    reservoirs["next_publication_date"] == year_one_date
).sum()

print(f"Completely duplicated rows: {reservoirs.duplicated().sum()}")

display(column_summary)

# Area numbers are reused, so the cross-tabulation shows which number belongs
# to each area type in the dataset.
display(pd.crosstab(
    reservoirs["area_type"],
    reservoirs["area_number"],
    margins=True
))

### Investigating the unusual year-1 date

I sort the complete dataset by observation date to see exactly where the
year-1 value stops appearing. The chronological table and visualization make
the introduction of the publication-date variable visible.

In [ ]:
# dato_Id was renamed to observation_date earlier. I sort the complete dataset
# by this column so the weekly sequence becomes visible. Area type and area
# number are used as secondary sorting columns to keep each week organized.
sorted_reservoirs = (
    reservoirs
    .sort_values(["observation_date", "area_type", "area_number"])
    .reset_index(drop=True)
)

# The publication date is repeated for the nine geographical rows belonging
# to the same observation date. This two-column version helps me locate the
# point where the date value changes.
sorted_publication_dates = (
    sorted_reservoirs[["observation_date", "next_publication_date"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# I locate the first row where the year-1 value is no longer present.
first_regular_date = (
    sorted_publication_dates["next_publication_date"]
    .ne(year_one_date)
    .idxmax()
)

# I first show the first two complete weeks in the sorted dataset. This is
# enough to confirm that the observations begin in 1995 and include nine
# geographical rows per week.
columns_to_show = [
    "observation_date",
    "next_publication_date",
    "area_type",
    "area_number",
    "filling_fraction"
]

print("Beginning of the dataset after sorting by observation date:")
with pd.option_context("display.max_rows", None):
    display(sorted_reservoirs.loc[:17, columns_to_show])

# I then create a second view around the point where the publication-date
# value changes. Two weeks before and two weeks after give 45 actual rows.
first_changed_observation = sorted_publication_dates.loc[
    first_regular_date, "observation_date"
]
start_date = (
    pd.to_datetime(first_changed_observation) - pd.Timedelta(weeks=2)
).strftime("%Y-%m-%d")
end_date = (
    pd.to_datetime(first_changed_observation) + pd.Timedelta(weeks=2)
).strftime("%Y-%m-%d")

sorted_date_window = sorted_reservoirs.loc[
    sorted_reservoirs["observation_date"].between(start_date, end_date),
    columns_to_show
]

print("Rows around the change in next_publication_date:")
with pd.option_context("display.max_rows", None):
    display(sorted_date_window)

In [ ]:
# I only include actual measurements here. Identifiers such as area number,
# ISO year, and ISO week would not be useful in this distribution check.
measurement_columns = [
    "filling_fraction",
    "capacity_twh",
    "stored_energy_twh",
    "previous_week_filling_fraction",
    "weekly_filling_change"
]

measurement_labels = {
    "filling_fraction": "Filling fraction",
    "capacity_twh": "Capacity (TWh)",
    "stored_energy_twh": "Stored energy (TWh)",
    "previous_week_filling_fraction": "Previous-week filling fraction",
    "weekly_filling_change": "Weekly filling change"
}

# These calculations check whether the measurement columns follow the
# relationships suggested by their names. I keep this compact because the
# measurements are visualized separately later in the notebook.
energy_difference = (
    reservoirs["stored_energy_twh"]
    - reservoirs["capacity_twh"] * reservoirs["filling_fraction"]
).abs().max()

weekly_change_difference = (
    reservoirs["weekly_filling_change"]
    - (reservoirs["filling_fraction"]
       - reservoirs["previous_week_filling_fraction"])
).abs().max()

print(f"Largest energy calculation difference: {energy_difference:.8f} TWh")
print(f"Largest weekly-change difference: {weekly_change_difference:.8f}")
print(
    "Filling fractions above 100%:",
    (reservoirs["filling_fraction"] > 1).sum()
)

In [ ]:
# I use the sorted table from the previous cell and convert only the dates
# that are later than year 1. The year-1 value stays untouched.
publication_pattern = sorted_publication_dates.copy()
publication_pattern["observation_date"] = pd.to_datetime(
    publication_pattern["observation_date"]
)
publication_pattern["date_group"] = (
    publication_pattern["next_publication_date"] != year_one_date
)

later_dates = publication_pattern[publication_pattern["date_group"]].copy()
later_dates["next_publication_date"] = pd.to_datetime(
    later_dates["next_publication_date"]
)
later_dates["days_between_dates"] = (
    later_dates["next_publication_date"]
    - later_dates["observation_date"]
).dt.total_seconds() / (24 * 60 * 60)

first_later_date = later_dates["observation_date"].min()
example_weeks = later_dates.head(12)

fig, axes = plt.subplots(
    2, 1, figsize=(13, 9),
    gridspec_kw={"height_ratios": [1, 2]}
)

# The first panel shows the exact point where the values change.
axes[0].scatter(
    publication_pattern["observation_date"],
    publication_pattern["date_group"].astype(int),
    s=10
)
axes[0].axvline(
    first_later_date, color="darkorange", linestyle="--",
    label=f"Change: {first_later_date:%d.%m.%Y}"
)
axes[0].set_yticks([0, 1], ["Year-1 value", "Later date value"])
axes[0].set_title("Publication-date values after sorting by observation date")
axes[0].set_xlabel("Observation date")
axes[0].set_ylabel("Value found in the CSV")
axes[0].legend()
axes[0].grid(axis="x", alpha=0.3)

# The second panel connects the first twelve observation dates after the
# change to the corresponding next-publication dates.
for position, (_, row) in enumerate(example_weeks.iterrows()):
    axes[1].plot(
        [row["observation_date"], row["next_publication_date"]],
        [position, position], color="grey", linewidth=2
    )
    axes[1].scatter(
        row["observation_date"], position, color="steelblue",
        label="Observation date" if position == 0 else None
    )
    axes[1].scatter(
        row["next_publication_date"], position, color="darkorange",
        label="Next publication date" if position == 0 else None
    )
    middle = row["observation_date"] + (
        row["next_publication_date"] - row["observation_date"]
    ) / 2
    axes[1].text(
        middle, position - 0.18,
        f'{row["days_between_dates"]:.1f} days',
        ha="center", fontsize=8
    )

axes[1].set_yticks(
    range(len(example_weeks)),
    example_weeks["observation_date"].dt.strftime("%d.%m.%Y")
)
axes[1].invert_yaxis()
axes[1].set_title("First 12 date pairs after the change")
axes[1].set_xlabel("Calendar date")
axes[1].set_ylabel("Observation date")
axes[1].legend()
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

# This table counts the intervals for all date pairs after the change.
display(
    later_dates["days_between_dates"]
    .value_counts().sort_index()
    .rename_axis("days_between_dates")
    .reset_index(name="number_of_weeks")
)

### Preparing the publication date for later use

After I showed the sorted data to the lecturer, he confirmed that the sudden
change in February 2019 marks the introduction of a new variable. The year-1
value is therefore not a genuine publication date.

I keep the imported DataFrame unchanged so I can document what the CSV originally
contained. For the rest of the project, I create a cleaned copy where the year-1
value is replaced with `NaT`, Pandas' missing value for dates. This keeps the
column suitable for sorting and filtering. In the Streamlit app, I will display
these values as ‘Not recorded – column introduced in 2019’ because that is more
helpful to a user than either the year-1 date or `NaT`.

In [ ]:
# I create a cleaned copy so the imported data remains available for comparison.
reservoirs_clean = reservoirs.copy()

# I convert the observation date here as well, so both date columns are ready
# for sorting, filtering, and plotting later in the project.
reservoirs_clean["observation_date"] = pd.to_datetime(
    reservoirs_clean["observation_date"]
)

# Based on the discussion with the lecturer, I replace the year-1 value with
# Pandas' missing datetime value in the cleaned dataset.
reservoirs_clean["next_publication_date"] = (
    reservoirs_clean["next_publication_date"]
    .replace(year_one_date, pd.NaT)
)

# The remaining publication dates can now be stored as actual datetime values.
reservoirs_clean["next_publication_date"] = pd.to_datetime(
    reservoirs_clean["next_publication_date"]
)

print(
    "Missing publication dates:",
    reservoirs_clean["next_publication_date"].isna().sum()
)

# I show a few unique dates around the change to confirm that the earlier
# values are now NaT and the later recorded dates are preserved.
clean_date_check = (
    reservoirs_clean[["observation_date", "next_publication_date"]]
    .drop_duplicates()
    .sort_values("observation_date")
    .reset_index(drop=True)
)

display(
    clean_date_check.iloc[first_regular_date - 2:first_regular_date + 3]
)

### Interpretation of the date investigation

My first missing-value check reported no missing values because every cell
contained a stored value. However, the `next_publication_date` column contains
`0001-01-01T00:00:00` in 11,322 rows. Pandas reads this as an ordinary string,
so these rows are not counted as missing by `isna()`.

After sorting the dataset by `observation_date`, I found a clear change. The
year-1 value is present through 10 February 2019, while recorded publication
dates begin on 17 February 2019. The cleaned output confirms that the earlier
values are now represented by `NaT` without changing the later dates.

The dates recorded after the variable was introduced follow a clear pattern.
Of the 395 unique date pairs, 390 have an interval of 10 days and 13 hours
between the observation date and the next publication date. Four intervals are
11 days and 13 hours, while one is 12 days and 13 hours. I keep these recorded
dates unchanged.

### Checking the numerical columns

I also checked the relationships between the numerical measurements. Stored
energy corresponds
closely to capacity multiplied by filling fraction, while weekly change
corresponds to the difference between the current and previous filling
fractions.

Two filling fractions are slightly above 1.0, meaning slightly above 100%.
They are not repeated, and their related energy values remain mathematically
consistent. I therefore keep these observations unchanged.

## 4. Plotting each column separately

I use the cleaned DataFrame for the remaining visualizations. The publication
date has been prepared as a datetime column, while the measurement values remain
unchanged.

Not every column represents the same kind of information. I use frequency bars
for dates, categories, years, and weeks, and time-series lines for the reservoir
measurements. For the measurement plots I use the national total (`NO`) so that
different and overlapping geographical summaries are not joined into one line.
The fractions are formatted as percentages for readability, but their underlying
values are not changed.

In [ ]:
# The dataset contains several overlapping geographical groupings. I use the
# national total for the time-series plots so that each date appears only once.
national_data = (
    reservoirs_clean[reservoirs_clean["area_type"] == "NO"]
    .sort_values("observation_date")
)

# observation_date: number of national observations recorded per year
observations_per_year = (
    national_data["observation_date"]
    .dt.year
    .value_counts()
    .sort_index()
)

observations_per_year.plot(
    kind="bar",
    figsize=(12, 5),
    color="steelblue"
)
plt.title("Number of observations per year")
plt.xlabel("Year")
plt.ylabel("Number of weekly observations")
plt.tight_layout()
plt.show()


# area_type: number of rows belonging to each geographical grouping
reservoirs_clean["area_type"].value_counts().plot(
    kind="bar",
    figsize=(7, 4),
    color="seagreen"
)
plt.title("Observations by area type")
plt.xlabel("Area type")
plt.ylabel("Number of observations")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


# area_number: area numbers are reused across different area types
reservoirs_clean["area_number"].value_counts().sort_index().plot(
    kind="bar",
    figsize=(7, 4),
    color="darkorange"
)
plt.title("Observations by area number")
plt.xlabel("Area number")
plt.ylabel("Number of observations")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# iso_year: number of national observations in each ISO year
national_data["iso_year"].value_counts().sort_index().plot(
    kind="bar",
    figsize=(12, 5),
    color="mediumpurple"
)
plt.title("National observations by ISO year")
plt.xlabel("ISO year")
plt.ylabel("Number of weekly observations")
plt.tight_layout()
plt.show()


# iso_week: frequency of the different ISO week numbers
national_data["iso_week"].value_counts().sort_index().plot(
    kind="bar",
    figsize=(12, 5),
    color="slateblue"
)
plt.title("National observations by ISO week")
plt.xlabel("ISO week")
plt.ylabel("Number of observations")
plt.tight_layout()
plt.show()

### Reservoir measurements over time

The next cell creates one separate time-series figure for each measurement.
Keeping the figures separate preserves their original units and makes small
changes visible before the columns are compared in one combined plot.

In [ ]:
# Each measurement gets its own plot because the columns use different units
# and scales. The fraction columns are formatted as percentages without
# changing their underlying values.
measurement_plots = {
    "filling_fraction": {
        "title": "National reservoir filling level",
        "ylabel": "Filling level",
        "percentage": True
    },
    "capacity_twh": {
        "title": "National reservoir capacity",
        "ylabel": "Capacity (TWh)",
        "percentage": False
    },
    "stored_energy_twh": {
        "title": "Energy stored in Norwegian reservoirs",
        "ylabel": "Stored energy (TWh)",
        "percentage": False
    },
    "previous_week_filling_fraction": {
        "title": "Previous week's national filling level",
        "ylabel": "Previous filling level",
        "percentage": True
    },
    "weekly_filling_change": {
        "title": "Weekly change in national filling level",
        "ylabel": "Change in percentage points",
        "percentage": True
    }
}

for column, settings in measurement_plots.items():
    fig, axis = plt.subplots(figsize=(12, 5))

    axis.plot(
        national_data["observation_date"],
        national_data[column],
        color="steelblue",
        linewidth=1
    )

    axis.set_title(settings["title"])
    axis.set_xlabel("Observation date")
    axis.set_ylabel(settings["ylabel"])
    axis.grid(alpha=0.3)

    if settings["percentage"]:
        axis.yaxis.set_major_formatter(PercentFormatter(1.0))

    plt.tight_layout()
    plt.show()

### Interpretation of the separate plots

The yearly plots show close to one national observation per week. The final bar
for 2026 is lower because the CSV ends on 6 September 2026, not because the
earlier weeks are represented by missing values. `filling_fraction` and
`stored_energy_twh` show a strong annual cycle. The previous-week filling curve
has almost the same shape as the current filling curve, as expected from a
one-week shift. National capacity stays constant in this dataset, while weekly
change moves around zero and makes short increases and decreases easier to see.

## 5. Plotting the measurement columns together

I first plot all five reservoir measurements on their original scales. This is
important because it shows why a direct comparison is difficult: TWh values are
much larger than fractions and weekly changes. I then use min-max scaling for a
second comparison. Scaling changes the display to a common range from 0 to 1; it
does not change the source data.

I do not include dates, area codes, or ISO identifiers in the combined plot.
They describe when or where an observation belongs rather than measuring the
reservoirs, so plotting them on the same axis would not give a meaningful
comparison.

In [ ]:
# I deliberately start with the original values so the scale problem is visible.
fig, axis = plt.subplots(figsize=(13, 6))

for column in measurement_columns:
    axis.plot(
        national_data["observation_date"],
        national_data[column],
        label=measurement_labels[column],
        linewidth=1
    )

axis.set_title("National reservoir measurements on their original scales")
axis.set_xlabel("Observation date")
axis.set_ylabel("Original values (fractions and TWh)")
axis.legend(loc="upper left", ncol=2)
axis.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Min-max scaling requires variation, so I automatically leave out columns
# that are constant. National capacity is constant in this CSV.
changing_measurements = [
    column for column in measurement_columns
    if national_data[column].nunique() > 1
]

scaled_measurements = national_data[changing_measurements].apply(
    lambda column: (column - column.min()) / (column.max() - column.min())
)

fig, axis = plt.subplots(figsize=(13, 6))

for column in changing_measurements:
    axis.plot(
        national_data["observation_date"],
        scaled_measurements[column],
        label=measurement_labels[column],
        linewidth=1
    )

axis.set_title("National reservoir measurements after min-max scaling")
axis.set_xlabel("Observation date")
axis.set_ylabel("Scaled value (0 to 1)")
axis.legend(loc="upper left", ncol=2)
axis.grid(alpha=0.3)
plt.tight_layout()
plt.show()

excluded_columns = [
    column for column in measurement_columns
    if column not in changing_measurements
]
print("Excluded because the national series is constant:", excluded_columns)

### Interpretation of the combined plots

In the unscaled figure, capacity and stored energy dominate because they are
measured in TWh, while filling fractions and weekly change are much smaller
numbers. The first figure is therefore truthful about the original units but not
very useful for comparing shapes.

The scaled figure makes the patterns comparable. Current and previous-week
filling almost overlap, and stored energy follows the same seasonal pattern.
Weekly change behaves differently because it measures movement from one week to
the next rather than the reservoir level itself. `capacity_twh` is excluded from
the scaled figure because its national value is constant; min-max scaling would
otherwise require division by zero. The separate capacity plot still documents
that column.

## 6. Project documentation

### Work log (current draft)

I started the compulsory work by creating a project structure with separate
folders for the notebook, data, and future Streamlit pages. I installed the
packages from `requirements.txt` and configured the notebook to use the project's
Python 3.12 environment in VS Code. Selecting the correct kernel took some extra
work because VS Code initially continued to show an older Python version. Once
the environment was available as a Jupyter kernel, the notebook ran correctly.

I loaded `reservoirs.csv` with Pandas by using a relative path from the notebook
folder. This makes the project portable because another person can clone the
repository without changing a path tied to my computer. The file contains 14,877
rows and 11 columns. I inspected the original Norwegian headers, their data
types, and example values before renaming the headers to understandable English
names. Where an exact category definition could not be confirmed from the CSV,
I kept the original values instead of adding an unsupported translation.

The initial missing-value check reported no missing cells, but the publication
date contained `0001-01-01T00:00:00` in many historical rows. Sorting the data
revealed a sudden change on 17 February 2019. I showed this result to the
lecturer, who confirmed that it marked the introduction of a new variable. I
therefore kept the imported data unchanged but replaced the year-1 value with
`NaT` in a cleaned copy. This gives the column a proper datetime type and will
let me present a clearer explanation to users in the Streamlit app. I also
checked the relationships between the numerical columns. Two filling fractions
were slightly above 100%, so I documented them instead of silently removing
them.

For the visual exploration, I used bar charts for categorical and calendar
columns and time-series plots for the measurements. I selected the national
records for the time-series figures because combining the different area types
would mix overlapping summaries. I first plotted the
measurements together without scaling, which showed that the TWh columns dominate.
I then used min-max scaling to compare their patterns while keeping the original
data unchanged.

My Streamlit work is currently limited to setting up `main.py`, the `pages`
folder, the local data folder, and the dependency file. The actual four-page app
and deployment have not been completed yet. When that work is finished, I will
update this log with the navigation, caching, table, plotting controls, deployment
experience, and the final public Streamlit link.